# Phase 3: RAG Pipeline with LangChain

Build the full ingestion and retrieval system from scratch.

## 3.1 Document Ingestion (Offline Phase)

```
Documents -> Text Splitter -> Embeddings -> Vector Store
```

In [ ]:
import sys
sys.path.insert(0, '..')

# Step 1: Load documents
from src.ingestion.loader import load_all_documents

docs = load_all_documents()
print(f'Case law documents: {len(docs["case_law"])}')
print(f'Contract documents: {len(docs["contracts"])}')
print(f'\nSample document preview:')
print(docs['case_law'][0].page_content[:500])

In [ ]:
# Step 2: Split into chunks
from src.ingestion.splitter import split_documents

case_law_chunks = split_documents(docs['case_law'])
contract_chunks = split_documents(docs['contracts'])

print(f'\nSample chunk:')
print(f'Content: {case_law_chunks[0].page_content[:200]}...')
print(f'Metadata: {case_law_chunks[0].metadata}')

In [ ]:
# Step 3: Run full ingestion pipeline
from src.ingestion.ingest import run_ingestion

case_law_store, contracts_store = run_ingestion()

## 3.2 Retrieval + Generation (Online Phase)

```
User Query -> Embed -> Search Vector Store -> Top-K Results -> Augment Prompt -> LLM -> Answer
```

In [ ]:
# Test retrieval
from src.retrieval.retriever import search_case_law, search_contracts

query = 'Does COVID-19 qualify as force majeure?'
results = search_case_law(query, top_k=3)

print(f'Query: {query}\n')
for i, doc in enumerate(results, 1):
    print(f'Result {i} [{doc.metadata["doc_id"]}]:')
    print(f'{doc.page_content[:200]}...\n')

In [ ]:
# Full RAG query
from src.rag.chain import query_rag

result = query_rag(query, doc_type='case_law')
print('Answer:')
print(result['answer'])
print(f'\nSources: {len(result["sources_summary"])}')
for s in result['sources_summary']:
    print(f'  - {s["doc_id"]} ({s["doc_type"]})')